# Notebook 02: Natural Language Processing - Text Classification

**Objective:** Learn how to classify text into categories (sentiment, topic, intent) using pretrained transformer models.

**Prerequisites:** Python basics. Notebook 01 (Text Generation) is helpful but not required.

**Learning Objectives:**
- Understand how transformer-based text classification works
- Use the Pipeline API and direct model loading for sentiment analysis
- Apply zero-shot classification to categorize text without task-specific training
- Evaluate model predictions against a real dataset (SST-2)
- Recognize failure modes in sentiment classification

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU (Small)** | distilbert-base-uncased-finetuned-sst-2-english | 268MB | 2GB | 4GB RAM, CPU | Fast, accurate |
| **GPU (Medium)** | bert-base-uncased | 440MB | 4GB | 6GB VRAM (RTX 4080) | More versatile |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `datasets`, `pandas`
- See `requirements.txt` for full list

## Overview

**Text classification** assigns one or more predefined labels to a piece of text. Unlike text generation (Notebook 01), which produces new text, classification maps input text to a fixed set of categories.

**How it works:**
1. Text is tokenized into subword tokens
2. The transformer encoder processes all tokens in parallel, producing contextual representations
3. A classification head (a linear layer) maps the `[CLS]` token representation to class probabilities
4. The class with the highest probability is the prediction

The key difference from generation models (GPT-2) is the architecture: classification uses an **encoder** (BERT, DistilBERT) that reads the entire input at once, while generation uses a **decoder** that reads left-to-right.

**Common classification tasks:**
- **Sentiment analysis**: Positive / Negative / Neutral
- **Topic classification**: Sports, Politics, Technology, etc.
- **Spam detection**: Spam / Not spam
- **Intent recognition**: For chatbots and virtual assistants

## Expected Behaviors

### First Time Running
- **Model download**: ~268MB for DistilBERT (1-3 minutes)
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

### Classification Output Format
```python
[{'label': 'POSITIVE', 'score': 0.9998}]
```
- **label**: `POSITIVE` or `NEGATIVE` for sentiment analysis
- **score**: Confidence between 0 and 1 (higher = more confident)

### Expected Accuracy
- **Clear sentiment** ("I love this!"): 95-99% confidence
- **Neutral text** ("It's okay"): 50-70% confidence
- **Mixed sentiment**: Model picks the dominant tone

### Performance
- **CPU (single text)**: 20-50ms
- **CPU (batch of 30)**: 2-5 seconds
- **GPU**: 5-10x faster

## Setup and Installation

We import all libraries, set the random seed for reproducibility, and check the hardware environment.

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    set_seed,
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

We use **DistilBERT fine-tuned on SST-2**, a distilled version of BERT that is 60% faster and 40% smaller while retaining 97% of BERT's accuracy on sentiment analysis. The model was already fine-tuned on the Stanford Sentiment Treebank, so it works out of the box for positive/negative classification.

If you have a GPU, you can try `bert-base-uncased` for comparison, though note that the base model needs fine-tuning for classification tasks.

In [ ]:
# CHOOSE YOUR MODEL:

# Option 1: CPU-friendly (recommended for beginners)
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"  # 268MB, sentiment analysis

# Option 2: GPU-optimized (uncomment if you have RTX 4080 or similar)
# MODEL_NAME = "bert-base-uncased"  # 440MB, needs fine-tuning for specific tasks

print(f"Selected model: {MODEL_NAME}")

---

## Method 1: Using Pipeline (Simplest)

The `sentiment-analysis` pipeline wraps tokenization, model inference, and label decoding into a single call. This is identical in structure to the text generation pipeline from Notebook 01, but the underlying model is an encoder (BERT) instead of a decoder (GPT-2).

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    classifier = pipeline(
        "sentiment-analysis",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Check your internet connection for the first download.")
    raise

### Basic Sentiment Analysis

Let's classify a single text. The model returns a label (`POSITIVE` or `NEGATIVE`) and a confidence score.

In [ ]:
text = "I absolutely love this product! It exceeded all my expectations."

result = classifier(text)

print(f"Text: {text}")
print(f"Prediction: {result[0]['label']}")
print(f"Confidence: {result[0]['score']:.4f}")

### Batch Classification

Passing a list of texts is more efficient than classifying one at a time, because the model can process them in a single forward pass (with padding). Let's classify several texts at once and organize the results in a DataFrame.

In [ ]:
def classify_batch(
    texts: list[str],
    clf: pipeline = None,
) -> pd.DataFrame:
    """Classify a batch of texts and return results as a DataFrame.

    Args:
        texts: List of input texts to classify.
        clf: Sentiment analysis pipeline. Uses the global `classifier` if None.

    Returns:
        DataFrame with columns: Text, Label, Confidence.
    """
    if clf is None:
        clf = classifier

    results = clf(texts)
    rows = [
        {
            "Text": text[:60] + "..." if len(text) > 60 else text,
            "Label": res["label"],
            "Confidence": round(res["score"], 4),
        }
        for text, res in zip(texts, results)
    ]
    return pd.DataFrame(rows)


texts = [
    "This is the worst experience I've ever had.",
    "The movie was okay, nothing special.",
    "Absolutely fantastic! Highly recommend!",
    "I'm not sure how I feel about this.",
    "Terrible service and poor quality.",
]

batch_df = classify_batch(texts)
batch_df

Notice how confidence varies. Strongly worded texts ("worst ever", "absolutely fantastic") get high confidence, while ambiguous or neutral texts ("okay", "not sure") get lower scores. This is expected -- the model is well-calibrated.

---

## Method 2: Using Model and Tokenizer Directly (Advanced)

Loading the model and tokenizer separately gives us access to the raw logits and probabilities. This is useful when you need to inspect the model's internal representations, compute custom metrics, or integrate classification into a larger pipeline.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Number of labels: {model.config.num_labels}")
print(f"Label mapping: {model.config.id2label}")

With the model loaded, we can see the full probability distribution across all classes. For binary sentiment analysis, the two probabilities always sum to 1.

In [ ]:
def classify_with_probabilities(
    text: str,
    tok: AutoTokenizer = None,
    mdl: AutoModelForSequenceClassification = None,
) -> dict[str, float]:
    """Classify text and return probabilities for all classes.

    Args:
        text: Input text to classify.
        tok: Tokenizer instance. Uses global `tokenizer` if None.
        mdl: Model instance. Uses global `model` if None.

    Returns:
        Dictionary mapping class labels to their probabilities.
    """
    if tok is None:
        tok = tokenizer
    if mdl is None:
        mdl = model

    inputs = tok(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = mdl(**inputs)
        probabilities = F.softmax(outputs.logits, dim=-1)[0]

    return {
        mdl.config.id2label[i]: round(prob.item(), 4)
        for i, prob in enumerate(probabilities)
    }


text = "The customer support was incredibly helpful and responsive."
probs = classify_with_probabilities(text)

print(f"Text: {text}\n")
for label, prob in probs.items():
    print(f"  {label}: {prob:.4f}")

---

## Practical Applications

Let's apply sentiment analysis to realistic scenarios. Each example demonstrates a different use pattern.

### Example 1: Product Review Analysis

A common business application: analyze a batch of product reviews and compute aggregate sentiment statistics.

In [ ]:
def analyze_reviews(
    reviews: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Analyze a batch of product reviews for sentiment.

    Args:
        reviews: List of review texts.

    Returns:
        Tuple of (detailed results DataFrame, summary statistics DataFrame).
    """
    results = classifier(reviews)

    detail_rows = [
        {
            "Review": rev[:50] + "..." if len(rev) > 50 else rev,
            "Sentiment": res["label"],
            "Confidence": round(res["score"], 3),
        }
        for rev, res in zip(reviews, results)
    ]
    detail_df = pd.DataFrame(detail_rows)

    positive_count = sum(1 for r in results if r["label"] == "POSITIVE")
    negative_count = len(results) - positive_count
    avg_confidence = np.mean([r["score"] for r in results])

    summary_df = pd.DataFrame([{
        "Total Reviews": len(reviews),
        "Positive": positive_count,
        "Negative": negative_count,
        "Positive %": f"{positive_count / len(reviews):.1%}",
        "Avg Confidence": round(avg_confidence, 3),
    }])

    return detail_df, summary_df


reviews = [
    "Great product! Works exactly as described.",
    "Disappointed with the quality. Would not recommend.",
    "Decent for the price, but could be better.",
    "Exceeded my expectations! Will buy again.",
    "Arrived damaged and customer service was unhelpful.",
]

detail_df, summary_df = analyze_reviews(reviews)
print("=== Detailed Results ===")
display(detail_df)
print("\n=== Summary ===")
display(summary_df)

### Example 2: Social Media Monitoring

Tracking sentiment across social media posts. Note how informal language, emojis, and exclamation marks still work reasonably well -- DistilBERT was trained on diverse text data.

In [ ]:
posts = [
    "Just launched our new feature! So excited to share this with everyone!",
    "Another day, another bug. This is getting frustrating.",
    "Thanks for the amazing support team! Issue resolved quickly.",
    "Can't believe how slow the app has become lately.",
    "Love the new update! Everything runs so smoothly now.",
]

social_df = classify_batch(posts)
social_df

### Example 3: Confidence-Based Filtering

In production systems, you often want to only act on high-confidence predictions and route uncertain ones to human review. Let's build a function that separates predictions by confidence threshold.

In [ ]:
def filter_by_confidence(
    texts: list[str],
    threshold: float = 0.9,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split texts into high-confidence and uncertain predictions.

    Args:
        texts: List of input texts.
        threshold: Minimum confidence to count as high-confidence.

    Returns:
        Tuple of (high-confidence DataFrame, uncertain DataFrame).
    """
    results = classifier(texts)
    high_conf_rows: list[dict] = []
    uncertain_rows: list[dict] = []

    for text, res in zip(texts, results):
        row = {
            "Text": text[:50] + "..." if len(text) > 50 else text,
            "Label": res["label"],
            "Confidence": round(res["score"], 4),
        }
        if res["score"] >= threshold:
            high_conf_rows.append(row)
        else:
            uncertain_rows.append(row)

    return pd.DataFrame(high_conf_rows), pd.DataFrame(uncertain_rows)


mixed_texts = [
    "I'm having the best day ever!",
    "This is completely unacceptable.",
    "It's fine, I guess.",
    "Not bad, not great.",
    "Absolutely wonderful experience!",
    "Meh.",
]

confident_df, uncertain_df = filter_by_confidence(mixed_texts, threshold=0.9)
print(f"High-confidence ({len(confident_df)} texts):")
display(confident_df)
print(f"\nUncertain -- route to human review ({len(uncertain_df)} texts):")
display(uncertain_df)

---

## Zero-Shot Classification

What if you want to classify text into categories the model was never explicitly trained on? **Zero-shot classification** uses a natural language inference (NLI) model to check whether a text "entails" each candidate label. This means you can classify into *any* set of categories without any training.

This downloads a larger model (~1.6GB for BART-MNLI) but is extremely versatile.

In [ ]:
def zero_shot_classify(
    texts: list[str],
    labels: list[str],
) -> pd.DataFrame:
    """Classify texts into arbitrary categories using zero-shot inference.

    Args:
        texts: List of texts to classify.
        labels: Candidate category labels (can be any strings).

    Returns:
        DataFrame with text, predicted label, and confidence.
    """
    rows: list[dict] = []
    for text in texts:
        result = zero_shot_clf(text, labels)
        rows.append({
            "Text": text[:50] + "..." if len(text) > 50 else text,
            "Predicted": result["labels"][0],
            "Confidence": round(result["scores"][0], 4),
            "Runner-up": result["labels"][1],
        })
    return pd.DataFrame(rows)


try:
    print("Loading zero-shot classification model (facebook/bart-large-mnli)...")
    zero_shot_clf = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0 if torch.cuda.is_available() else -1,
    )
    print("Model loaded!\n")

    texts = [
        "I love playing basketball and watching NBA games.",
        "The new iPhone has an incredible camera system.",
        "Congress passed the infrastructure bill yesterday.",
        "The stock market reached an all-time high today.",
    ]
    labels = ["sports", "technology", "politics", "finance", "entertainment"]

    zs_df = zero_shot_classify(texts, labels)
    display(zs_df)

except Exception as e:
    print(f"Could not load zero-shot model: {e}")
    print("This requires ~1.6GB download. Skipping.")

The power of zero-shot classification is that you can change the labels to *anything* without retraining. Try replacing the labels with `["urgent", "normal", "low priority"]` for ticket triage, or `["positive", "negative", "neutral"]` for three-way sentiment.

---

## Dataset Evaluation: SST-2

Let's evaluate our model against a real benchmark dataset. The **Stanford Sentiment Treebank (SST-2)** contains ~67k training and 872 validation movie review sentences labeled as positive or negative. This is the same dataset the model was fine-tuned on, so we expect high accuracy.

In [ ]:
def evaluate_on_sst2(
    num_samples: int = 50,
) -> tuple[pd.DataFrame, float]:
    """Evaluate the classifier on SST-2 validation set.

    Args:
        num_samples: Number of validation samples to evaluate.

    Returns:
        Tuple of (sample results DataFrame, overall accuracy).
    """
    from datasets import load_dataset

    print("Loading SST-2 dataset...")
    dataset = load_dataset("sst2", split="validation")
    print(f"Loaded {len(dataset)} validation examples")

    texts = dataset["sentence"][:num_samples]
    true_labels = dataset["label"][:num_samples]
    results = classifier(texts)

    label_map = {0: "NEGATIVE", 1: "POSITIVE"}
    correct = 0
    rows: list[dict] = []

    for text, true_label, pred in zip(texts, true_labels, results):
        true_str = label_map[true_label]
        is_correct = pred["label"] == true_str
        correct += int(is_correct)
        rows.append({
            "Text": text[:50] + "..." if len(text) > 50 else text,
            "True Label": true_str,
            "Predicted": pred["label"],
            "Confidence": round(pred["score"], 4),
            "Correct": is_correct,
        })

    accuracy = correct / num_samples
    return pd.DataFrame(rows), accuracy


try:
    eval_df, accuracy = evaluate_on_sst2(num_samples=50)
    print(f"\nAccuracy on 50 samples: {accuracy:.2%}\n")
    print("Sample predictions (first 10):")
    display(eval_df.head(10))
except Exception as e:
    print(f"Could not load dataset: {e}")
    print("Install with: pip install datasets")

---

## Performance Benchmarking

Let's measure classification speed across different batch sizes. Batching is significantly faster than classifying texts one at a time because the model can parallelize computation across inputs.

In [ ]:
def benchmark_classification(
    batch_sizes: list[int] | None = None,
    num_runs: int = 3,
) -> pd.DataFrame:
    """Benchmark classification speed across different batch sizes.

    Args:
        batch_sizes: List of batch sizes to test. Defaults to [1, 5, 10, 20].
        num_runs: Number of runs to average per batch size.

    Returns:
        DataFrame with batch size, avg time, and throughput.
    """
    if batch_sizes is None:
        batch_sizes = [1, 5, 10, 20]

    sample_text = "This product is absolutely wonderful and I love it."
    rows: list[dict] = []

    for batch_size in batch_sizes:
        batch = [sample_text] * batch_size
        times: list[float] = []
        for _ in range(num_runs):
            start = time.time()
            classifier(batch)
            times.append(time.time() - start)

        avg_time = np.mean(times)
        rows.append({
            "Batch Size": batch_size,
            "Avg Time (s)": round(avg_time, 3),
            "Texts/sec": round(batch_size / avg_time, 1),
            "Device": "GPU" if torch.cuda.is_available() else "CPU",
        })

    return pd.DataFrame(rows)


print(f"Benchmarking {MODEL_NAME} on {device}...\n")
bench_df = benchmark_classification()
bench_df

Throughput (texts per second) increases with batch size because the model amortizes its fixed overhead across more inputs. In production, finding the optimal batch size for your hardware is one of the easiest performance wins.

---

## Error Analysis: Where Sentiment Models Fail

Sentiment analysis models have well-known failure modes. Understanding these is critical for building reliable systems. Let's test cases that are designed to expose common weaknesses.

In [ ]:
def analyze_failure_modes(
    test_cases: list[tuple[str, str, str]],
) -> pd.DataFrame:
    """Test the classifier on cases designed to expose failure modes.

    Args:
        test_cases: List of (text, expected_label, failure_category) tuples.

    Returns:
        DataFrame showing category, text, expected vs predicted label, and correctness.
    """
    rows: list[dict] = []
    for text, expected, category in test_cases:
        result = classifier(text)[0]
        rows.append({
            "Category": category,
            "Text": text[:55] + "..." if len(text) > 55 else text,
            "Expected": expected,
            "Predicted": result["label"],
            "Confidence": round(result["score"], 3),
            "Correct": result["label"] == expected,
        })
    return pd.DataFrame(rows)


test_cases = [
    # Sarcasm -- model often misses verbal irony
    ("Oh great, another meeting that could have been an email.", "NEGATIVE", "Sarcasm"),
    ("What a surprise, the software crashed again.", "NEGATIVE", "Sarcasm"),
    # Negation -- flipping sentiment with 'not'
    ("This is not a good product.", "NEGATIVE", "Negation"),
    ("I don't think this is bad at all.", "POSITIVE", "Negation"),
    # Mixed sentiment -- both positive and negative signals
    ("The food was amazing but the service was terrible.", "NEGATIVE", "Mixed"),
    ("Great features, but way too expensive.", "NEGATIVE", "Mixed"),
    # Neutral / factual -- no sentiment intended
    ("The meeting is scheduled for 3pm tomorrow.", "NEUTRAL", "Neutral"),
    ("The product weighs 2.5 kilograms.", "NEUTRAL", "Neutral"),
]

print("Failure Mode Analysis:\n")
failure_df = analyze_failure_modes(test_cases)
failure_df

**What to notice:**

- **Sarcasm**: The model reads surface-level positive words ("great", "surprise") without detecting irony. This is a known limitation of all current sentiment models.
- **Negation**: Simple negation ("not good") is sometimes handled, but complex double negation ("don't think this is bad") often confuses the model.
- **Mixed sentiment**: The model picks a side rather than recognizing that both positive and negative signals are present. A three-way model (positive/negative/neutral) would help here.
- **Neutral text**: This model has no neutral class -- it forces every text into positive or negative. Binary classification is a fundamental limitation. For production use, consider a model with a neutral class or use confidence thresholds to flag uncertain predictions.

---

## Exercises

1. **Custom dataset**: Create a list of 10+ texts from a domain you care about (app reviews, emails, tweets). Run `classify_batch()` and compute the positive/negative split.

2. **Confidence threshold tuning**: Use `filter_by_confidence()` with thresholds of 0.7, 0.8, 0.9, and 0.95. Plot (or print) how many texts fall into the "uncertain" bucket at each threshold.

3. **Zero-shot custom labels**: Use the zero-shot classifier with labels relevant to your work (e.g., `["bug report", "feature request", "question"]` for support tickets).

4. **Emotion detection**: Try `cardiffnlp/twitter-roberta-base-emotion` for multi-class emotion classification (joy, sadness, anger, etc.).

5. **Full SST-2 evaluation**: Call `evaluate_on_sst2(num_samples=872)` to evaluate on the entire validation set. What is the overall accuracy?

---

## State-of-the-Art Open Models (Reference)

| Model | Developer | Strengths | Size | Best For |
|-------|-----------|-----------|------|----------|
| **RoBERTa** | Meta | Robustly optimized BERT, outperforms BERT | 125-355M | High-accuracy classification |
| **DeBERTa-V3** | Microsoft | Best NLU benchmark scores | 140-434M | When accuracy is critical |
| **ELECTRA** | Google | More efficient pretraining | 14-335M | Limited compute budgets |
| **Twitter-RoBERTa** | Cardiff NLP | Trained on 58M tweets | 125M | Social media analysis |
| **FinBERT** | ProsusAI | Financial sentiment | 110M | Finance domain |
| **BioBERT** | DMIS Lab | Biomedical literature | 110M | Medical text |

**Benchmarks (GLUE):** DistilBERT ~86, BERT-base ~88, RoBERTa-large ~91, DeBERTa-V3 ~92

For domain-specific tasks, a specialized model (FinBERT, BioBERT) will almost always outperform a general-purpose model, even one that's larger.

---

## Key Takeaways

- **Text classification** maps input text to a fixed set of categories using an encoder model (BERT/DistilBERT)
- **Batch processing** is significantly faster than classifying one text at a time
- **Confidence scores** are well-calibrated: use thresholds to route uncertain predictions to human review
- **Zero-shot classification** lets you classify into arbitrary categories without retraining
- **Failure modes** include sarcasm, negation, mixed sentiment, and the lack of a neutral class

## Next Steps

- **Notebook 03: Text Summarization** -- Condense long documents into short summaries
- **Notebook 04-05 (NLP): Fine-tuning** -- Train your own classifier on custom categories
- [HuggingFace Text Classification Models](https://huggingface.co/models?pipeline_tag=text-classification)

## Resources

- [Text Classification Guide](https://huggingface.co/docs/transformers/tasks/sequence_classification)
- [Sentiment Analysis Tutorial](https://huggingface.co/blog/sentiment-analysis-python)
- [Zero-Shot Classification](https://huggingface.co/tasks/zero-shot-classification)